# **03 · Optimización de hiperparámetros con Optuna**

Busca los hiperparámetros que mejoran al modelo base del notebook 02.

**¿Por qué Optuna y no Grid Search?** Cada prueba es un entrenamiento completo (varios minutos de GPU). Una
grilla de 5 hiperparámetros con 3 valores cada uno serían 243 entrenamientos. Optuna usa búsqueda bayesiana
(TPE): con los resultados de las pruebas anteriores elige qué valores probar después, y llega a una buena
zona con muchas menos pruebas.

| Elemento | Definición |
|---|---|
| **Métrica que se maximiza** | mAP50-95 sobre las 10 fotos propias de validación |
| **Métricas que se registran** | precisión, recobrado, F1, mAP50 y mAP50-95 |
| **Pruebas** | 12 entrenamientos de 30 épocas como máximo (la primera, con los valores del modelo base) |
| **Salida** | `results/optuna_pruebas.csv` (una fila por prueba) y `results/mejores_hiperparametros.json` |

> **Antes de empezar (Colab):** GPU (T4) y haber corrido el notebook 01. La búsqueda tarda unas 2 horas y el
> entrenamiento final, unos 15 minutos. Si Colab se desconecta, volver a correr todo: la búsqueda **continúa**
> desde la última prueba terminada.

## **1. Preparación del entorno**

In [ ]:
!pip -q install "ultralytics==8.4.153" optuna

In [ ]:
# Carpetas del proyecto. En Colab todo queda en Drive (carpeta limon_ia), con la misma estructura
# que el repositorio: data/, results/ y runs/ (modelos entrenados). Fuera de Colab se usa el repositorio.
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive/limon_ia")   # carpeta del proyecto en Drive
    TRABAJO = Path("/content")                        # disco de Colab: rápido, pero se borra al cerrar
except ImportError:
    BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    TRABAJO = BASE / "data"

DATA, RESULTS, RUNS = BASE / "data", BASE / "results", BASE / "runs"
for carpeta in (DATA, RESULTS, RUNS):
    carpeta.mkdir(parents=True, exist_ok=True)
DATOS = TRABAJO / "datos"                             # dataset unificado, descomprimido
print("Proyecto en", BASE)

In [ ]:
import json
import shutil

import optuna
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from ultralytics import YOLO

IMGSZ = 640            # tamaño de imagen, igual que el modelo base
N_PRUEBAS = 12         # cantidad de entrenamientos de la búsqueda
EPOCAS_PRUEBA = 30     # épocas máximas de cada prueba (el modelo base llegó a su mejor época cerca de la 30)
PACIENCIA_PRUEBA = 10  # parada temprana de cada prueba
EPOCAS_FINAL = 60      # épocas máximas del entrenamiento final, igual que el modelo base
PACIENCIA_FINAL = 20

In [ ]:
# --------
# Función que deja listo el dataset para entrenar.
#   - Descomprime data/datos_limon.zip (lo genera el notebook 01) si todavía no está en el disco.
#   - Repite "repetir" veces las fotos propias de entrenamiento (sobremuestreo), para que pesen más
#     frente a las 915 imágenes del dataset base.
#   - Devuelve: la ruta del archivo .yaml que le dice a YOLO dónde están train y val.
# --------
def preparar_datos(repetir=3):
    if not (DATOS / "propio").exists():
        shutil.unpack_archive(DATA / "datos_limon.zip", DATOS)
    rep = DATOS / "propio" / f"train_x{repetir}"
    if not rep.exists():
        for sub in ("images", "labels"):
            (rep / sub).mkdir(parents=True)
        for img in sorted((DATOS / "propio" / "train" / "images").iterdir()):
            etiqueta = DATOS / "propio" / "train" / "labels" / (img.stem + ".txt")
            for k in range(repetir):
                shutil.copy(img, rep / "images" / f"r{k}_{img.name}")
                shutil.copy(etiqueta, rep / "labels" / f"r{k}_{img.stem}.txt")
    # train: todo el dataset base + las fotos propias de train repetidas. val: las 10 fotos propias de valid.
    train = [str(DATOS / "base" / s / "images") for s in ("train", "valid", "test")
             if any((DATOS / "base" / s / "images").glob("*"))] + [str(rep / "images")]
    ruta = TRABAJO / f"limones_x{repetir}.yaml"
    ruta.write_text(yaml.safe_dump({"train": train, "val": str(DATOS / "propio" / "valid" / "images"),
                                    "nc": 1, "names": {0: "limon"}}))
    return str(ruta)


# --------
# Función que evalúa un modelo sobre una o más carpetas de imágenes.
#   - Recibe: el modelo, un nombre para la fila y los conjuntos propios a usar ("valid", "test").
#   - Devuelve: diccionario con precisión, recobrado, F1, mAP50 y mAP50-95.
# --------
def evaluar(modelo, nombre, conjuntos):
    carpetas = [str(DATOS / "propio" / c / "images") for c in conjuntos]
    ruta = TRABAJO / f"eval_{'_'.join(conjuntos)}.yaml"
    ruta.write_text(yaml.safe_dump({"train": carpetas, "val": carpetas, "nc": 1, "names": {0: "limon"}}))
    m = modelo.val(data=str(ruta), imgsz=IMGSZ, project=str(TRABAJO / "evals"), name=nombre, exist_ok=True,
                   verbose=False, plots=False)
    p, r = float(m.box.mp), float(m.box.mr)
    return {"modelo": nombre, "conjunto": "+".join(conjuntos),
            "imágenes": sum(len(list(Path(c).glob("*"))) for c in carpetas),
            "precisión": round(p, 3), "recobrado": round(r, 3), "F1": round(2 * p * r / (p + r + 1e-9), 3),
            "mAP50": round(float(m.box.map50), 3), "mAP50-95": round(float(m.box.map), 3)}

## **2. Espacio de búsqueda**

| Hiperparámetro | Qué controla | Rango | Valor base |
|---|---|---|---|
| `lr0` | Tasa de aprendizaje inicial: cuánto se corrigen los pesos en cada paso | 0,0003 a 0,01 (escala logarítmica) | 0,002 |
| `weight_decay` | Regularización: penaliza los pesos grandes para reducir el sobreajuste | 0,00005 a 0,005 (escala logarítmica) | 0,0005 |
| `hsv_h` | Aumento de datos: cuánto se varía al azar el **tono** de cada imagen | 0 a 0,08 | 0,015 |
| `scale` | Aumento de datos: cuánto se agranda o achica al azar cada imagen | 0,2 a 0,9 | 0,5 |
| `repetir` | Cuántas veces se repite cada foto propia de entrenamiento (sobremuestreo) | 1, 3 o 5 | 3 |

Los tres últimos se eligieron por lo que mostraron las features del notebook 01: `hsv_h` actúa sobre el tono,
`scale` sobre el tamaño de los limones y `repetir` sobre el peso de las fotos propias frente al dataset
público.

**Nota sobre el optimizador.** Con `optimizer="auto"` Ultralytics ignora `lr0` y elige todo solo: en el
modelo base eligió AdamW con tasa 0,002. Para que `lr0` y `weight_decay` tengan efecto, acá se fija
`optimizer="AdamW"`. Por eso la primera prueba usa los valores base con AdamW: es la referencia justa.

In [ ]:
VALORES_BASE = {"lr0": 0.002, "weight_decay": 0.0005, "hsv_h": 0.015, "scale": 0.5, "repetir": 3}


# --------
# Función que entrena un modelo con un conjunto de hiperparámetros.
#   - Recibe: los hiperparámetros, la carpeta y el nombre de la corrida, épocas máximas y paciencia.
#   - Devuelve: las métricas del mejor modelo (best.pt) en validación.
# --------
def entrenar(hp, proyecto, nombre, epocas, paciencia):
    modelo = YOLO("yolo26n.pt")
    m = modelo.train(
        data=preparar_datos(hp["repetir"]), epochs=epocas, patience=paciencia, imgsz=IMGSZ, batch=16, seed=0,
        optimizer="AdamW", lr0=hp["lr0"], weight_decay=hp["weight_decay"], hsv_h=hp["hsv_h"], scale=hp["scale"],
        project=str(proyecto), name=nombre, exist_ok=True, plots=False, verbose=False,
    )
    p, r = float(m.box.mp), float(m.box.mr)
    return {"precisión": p, "recobrado": r, "F1": 2 * p * r / (p + r + 1e-9),
            "mAP50": float(m.box.map50), "mAP50-95": float(m.box.map)}


# --------
# Función objetivo: Optuna la llama una vez por prueba.
#   - Recibe: trial, el objeto con el que se piden los valores a probar.
#   - Devuelve: el valor a maximizar (mAP50-95 en validación).
# --------
def objetivo(trial):
    hp = {
        "lr0": trial.suggest_float("lr0", 3e-4, 1e-2, log=True),
        "weight_decay": trial.suggest_float("weight_decay", 5e-5, 5e-3, log=True),
        "hsv_h": trial.suggest_float("hsv_h", 0.0, 0.08),
        "scale": trial.suggest_float("scale", 0.2, 0.9),
        "repetir": trial.suggest_categorical("repetir", [1, 3, 5]),
    }
    metricas = entrenar(hp, TRABAJO / "pruebas", f"prueba_{trial.number:02d}", EPOCAS_PRUEBA, PACIENCIA_PRUEBA)
    # Guardamos todas las métricas en la prueba y borramos sus pesos (no se usan y ocupan lugar).
    for nombre, valor in metricas.items():
        trial.set_user_attr(nombre, round(valor, 4))
    shutil.rmtree(TRABAJO / "pruebas" / f"prueba_{trial.number:02d}" / "weights", ignore_errors=True)
    return metricas["mAP50-95"]

## **3. Búsqueda**

El estudio se guarda en una base de datos (`results/optuna.db`) después de cada prueba, junto con el CSV. Si
la sesión se corta, al volver a correr esta celda se cargan las pruebas hechas y solo se corren las que faltan.

In [ ]:
# La base de datos se usa en el disco local (rápido) y se copia a results/ después de cada prueba.
BD_LOCAL, BD_GUARDADA = TRABAJO / "optuna.db", RESULTS / "optuna.db"
if BD_GUARDADA.exists() and not BD_LOCAL.exists():
    shutil.copy(BD_GUARDADA, BD_LOCAL)

estudio = optuna.create_study(study_name="limon_yolo26n", storage=f"sqlite:///{BD_LOCAL}", load_if_exists=True,
                              direction="maximize", sampler=optuna.samplers.TPESampler(seed=0))
if not estudio.trials:
    estudio.enqueue_trial(VALORES_BASE)        # la primera prueba: los valores del modelo base


# --------
# Función que arma la tabla de pruebas: una fila por prueba, con sus hiperparámetros y métricas.
# --------
def tabla_de_pruebas():
    filas = []
    for t in estudio.trials:
        if t.state == optuna.trial.TrialState.COMPLETE:
            filas.append({"prueba": t.number, **{hp: round(v, 6) for hp, v in t.params.items()}, **t.user_attrs,
                          "minutos": round((t.datetime_complete - t.datetime_start).total_seconds() / 60, 1)})
    columnas = ["prueba", *VALORES_BASE, "precisión", "recobrado", "F1", "mAP50", "mAP50-95", "minutos"]
    return pd.DataFrame(filas, columns=columnas)


# Después de cada prueba: guardamos el CSV y la base de datos.
def guardar(estudio, trial):
    tabla_de_pruebas().to_csv(RESULTS / "optuna_pruebas.csv", index=False)
    shutil.copy(BD_LOCAL, BD_GUARDADA)


hechas = len(tabla_de_pruebas())
print(f"Pruebas hechas: {hechas} de {N_PRUEBAS}")
if hechas < N_PRUEBAS:
    estudio.optimize(objetivo, n_trials=N_PRUEBAS - hechas, callbacks=[guardar], catch=(RuntimeError,))

pruebas = tabla_de_pruebas()
pruebas.to_csv(RESULTS / "optuna_pruebas.csv", index=False)
pruebas.sort_values("mAP50-95", ascending=False).round(4)

## **4. Resultados de la búsqueda**

A la izquierda, el valor de cada prueba y el mejor acumulado: muestra si la búsqueda siguió mejorando. A la
derecha, la métrica contra cada hiperparámetro: si los puntos altos se agrupan en una zona, ese
hiperparámetro importa; si quedan dispersos, influye poco.

In [ ]:
mejor = pruebas.loc[pruebas["mAP50-95"].idxmax()]
base = pruebas.loc[pruebas["prueba"] == 0].iloc[0] if (pruebas["prueba"] == 0).any() else None

fig, ejes = plt.subplots(1, 6, figsize=(24, 4), gridspec_kw={"width_ratios": [1.6, 1, 1, 1, 1, 1]})
ejes[0].plot(pruebas["prueba"], pruebas["mAP50-95"], "o", label="cada prueba")
ejes[0].plot(pruebas["prueba"], pruebas["mAP50-95"].cummax(), "-", label="mejor hasta el momento")
ejes[0].set_xlabel("prueba"); ejes[0].set_ylabel("mAP50-95 (validación)"); ejes[0].set_title("Historia de la búsqueda")
ejes[0].set_xticks(pruebas["prueba"]); ejes[0].legend()
for ax, hp in zip(ejes[1:], VALORES_BASE):
    ax.scatter(pruebas[hp], pruebas["mAP50-95"], c=pruebas["prueba"], cmap="viridis")
    ax.scatter([mejor[hp]], [mejor["mAP50-95"]], s=220, facecolors="none", edgecolors="red", label="mejor")
    if hp in ("lr0", "weight_decay"):
        ax.set_xscale("log")
    ax.set_title(hp)
ejes[1].legend()
plt.tight_layout()
plt.savefig(RESULTS / "fig_optuna.png", dpi=110)
plt.show()

# Tomamos los valores exactos de la mejor prueba desde el estudio.
mejores_hp = estudio.trials[int(mejor["prueba"])].params
(RESULTS / "mejores_hiperparametros.json").write_text(json.dumps(mejores_hp, indent=2))
print(f"Mejor prueba: la {int(mejor['prueba'])}, con mAP50-95 = {mejor['mAP50-95']:.3f}")
if base is not None:
    print(f"Valores base (prueba 0): mAP50-95 = {base['mAP50-95']:.3f} "
          f"-> diferencia {mejor['mAP50-95'] - base['mAP50-95']:+.3f}")
print(json.dumps(mejores_hp, indent=2))

**Cuidado al interpretar.** La validación tiene solo 10 fotos (58 limones): una diferencia de uno o dos
limones mueve la métrica en el segundo decimal. Mejoras chicas respecto de la prueba 0 pueden ser ruido y no
un efecto real del hiperparámetro. Por eso la comparación final se hace en el notebook 04 sobre las fotos de
test, que no se usaron para elegir nada.

## **5. Entrenamiento final con los mejores hiperparámetros**

Se entrena de nuevo, ahora con el mismo máximo de épocas que el modelo base, y se guarda en
`runs/optimizado/`. Si ya existe, no se vuelve a entrenar.

In [ ]:
mejor_opt = RUNS / "optimizado" / "weights" / "best.pt"
if mejor_opt.exists():
    print("Ya hay un modelo optimizado entrenado:", mejor_opt)
else:
    entrenar(mejores_hp, RUNS, "optimizado", EPOCAS_FINAL, PACIENCIA_FINAL)
shutil.copy(RUNS / "optimizado" / "results.csv", RESULTS / "optimizado_curvas.csv")
print("Curvas guardadas en", RESULTS / "optimizado_curvas.csv")

# Comparación preliminar en validación (la definitiva, en test, está en el notebook 04).
filas = [evaluar(YOLO(str(mejor_opt)), "optimizado", ["valid"])]
if (RUNS / "base" / "weights" / "best.pt").exists():
    filas.insert(0, evaluar(YOLO(str(RUNS / "base" / "weights" / "best.pt")), "base", ["valid"]))
comparacion = pd.DataFrame(filas)
comparacion.to_csv(RESULTS / "comparacion_validacion.csv", index=False)
comparacion

**Análisis (completar después de correr).** Anotar: qué prueba ganó y con qué valores, cuánto mejoró respecto
de la prueba 0, qué hiperparámetros parecen importar según los gráficos y si la mejora se mantiene en el
entrenamiento final.